# 05 — 双恒电位仪（BiStat）与 WE32 多通道阵列

本笔记本介绍两种多电极功能：

**BiStat** — 某些 Ivium 设备内置的第二工作电极（WE2）通道。WE1 和 WE2 共用同一参比和对电极，但可施加独立电位并测量独立电流。常用于：
- 旋转环盘电极（RRDE）实验
- 发生器-收集器实验
- 双电极流通池

**WE32** — 32 通道工作电极阵列多路复用器附件。一次 `read_we32_currents()` 调用即可同时测量所有 32 个通道。常用于：
- 筛选阵列（组合电化学）
- 腐蚀成像
- 传感器阵列

### 前提条件
- 驱动已打开，设备已连接（参见 `02_device_and_instance_management.ipynb`）
- **BiStat 部分**：设备必须支持 BiStat（例如带 BiStat 选项的 IviumStat）
- **WE32 部分**：必须连接 WE32 附件
- 错误处理参考：`01_getting_started.ipynb` — 第 4 节

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
Pyvium.connect_device()
print("就绪")

---

# A 部分 — BiStat（双恒电位仪）

## A1. BiStat 连接模式

使用连接模式 10 或 11 启用 BiStat：

| 代码 | 模式 |
|------|------|
| 10 | BiStat 4EL（4 电极，RRDE 默认） |
| 11 | BiStat 2EL（2 电极） |

在典型的 RRDE 设置中，WE1 为盘电极，WE2 为环电极。

In [ ]:
Pyvium.set_connection_mode(10)  # BiStat 4EL
print("已选择 BiStat 4EL 模式")

## A2. BiStat 模式和自动电流量程

`set_bistat_mode()` 同时控制两个关联设置：

| 值 | BiStat 扫描 | WE2 参考系 | 自动电流量程 |
|-------|----------------|---------------------|-------------|
| 0 | 标准（静态 WE2 偏移） | WE2 电位相对于 **RE** | 关闭 |
| 1 | 扫描（WE2 跟随 WE1 扫描） | WE2 电位相对于 **WE1** | 开启 |

- **模式 0（标准）：** WE2 保持在相对参比电极的固定偏移处。适用于环需要绝对电位的 RRDE（例如「将环保持在 +0.4 V vs. Ag/AgCl」）。
- **模式 1（扫描）：** WE2 跟随 WE1 保持固定偏移。适用于希望 WE2 在整个扫描过程中始终比 WE1 超前固定伏数的发生器-收集器实验。

> `connect_device()` 始终将自动电流量程重置为关闭（模式 0）。如需扫描模式，请在连接后显式调用 `set_bistat_mode(1)`。

In [ ]:
Pyvium.set_bistat_mode(0)  # 标准：固定 WE2 电位，自动电流量程关闭
print("BiStat 模式：标准（静态 WE2 电位）")

## A3. 设置 WE1 和 WE2 的电流量程

WE2（环）通常承载较小的电流分量，因此可使用更精细的量程。

WE2（BiStat）的量程代码：

| 代码 | 量程 |
|------|------|
| 0 | 10 mA |
| 1 | 1 mA |
| 2 | 100 µA |
| 3 | 10 µA |
| 4 | 1 µA |

In [ ]:
Pyvium.set_current_range(2)      # WE1（盘）：100 µA
Pyvium.set_we2_current_range(2)  # WE2（环）：100 µA
print("WE1 和 WE2 电流量程：100 µA")

## A4. 设置独立电位

- `set_potential(E)` — 设置 WE1 电位（相对参比电极）
- `set_we2_potential(E)` — 设置 WE2 电位；参考系取决于 BiStat 模式：
  - **模式 0**（标准）：值为相对 RE 的偏移
  - **模式 1**（扫描）：值为相对 WE1 的偏移

In [ ]:
WE1_POTENTIAL = 0.0   # V — 盘电位相对参比
WE2_OFFSET    = 0.4   # V — 环偏移相对 WE1

Pyvium.set_potential(WE1_POTENTIAL)
Pyvium.set_we2_potential(WE2_OFFSET)
print(f"WE1（盘）: {WE1_POTENTIAL} V")
print(f"WE2（环）: WE1 + {WE2_OFFSET} V = {WE1_POTENTIAL + WE2_OFFSET} V vs. ref")

## A5. 双通道测量（RRDE 示例）

两个通道独立读取。此示例模拟 RRDE 式保持：盘处于还原电位，环保持在氧化电位以检测产物。

In [ ]:
DURATION_S = 10.0
INTERVAL_S = 0.2

Pyvium.set_cell_on()
time.sleep(0.1)

timestamps, I_disk, I_ring = [], [], []
t_start = time.time()

while (time.time() - t_start) < DURATION_S:
    t   = time.time() - t_start
    I1  = Pyvium.get_current()      # WE1（盘）
    I2  = Pyvium.get_we2_current()  # WE2（环）
    timestamps.append(t)
    I_disk.append(I1 * 1e6)   # µA
    I_ring.append(I2 * 1e6)
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()

# 计算收集效率 N = -I_ring / I_disk（理想 RRDE）
avg_disk = np.mean(I_disk)
avg_ring = np.mean(I_ring)
if avg_disk != 0:
    N = -avg_ring / avg_disk
    print(f"环收集效率 N ≈ {N:.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(timestamps, I_disk, 'b-', label=f'WE1 盘（均值 {avg_disk:.2f} µA）')
ax.plot(timestamps, I_ring, 'r-', label=f'WE2 环（均值 {avg_ring:.2f} µA）')
ax.set_xlabel("时间 (s)")
ax.set_ylabel("电流 (µA)")
ax.set_title("BiStat — 双通道测量")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# B 部分 — WE32 多通道阵列

> 🔬 **未经测试 — 无可用硬件。** WE32 方法基于 IviumSoft DLL 规范实现，但未在实际 WE32 附件上验证。API 和代码结构预计是正确的，但参数范围、时序约束和返回值尚未在真实硬件上确认。欢迎 WE32 用户提供反馈。

WE32 附件将 32 个工作电极连接到单台恒电位仪。所有通道共用同一参比和对电极。

## B1. 选择活跃的 WE32 通道

`set_we32_channel(index)` 选择一个通道作为当前活跃电极（用于方法和直接模式）。这**不影响** `read_we32_currents()`，后者始终同时读取全部 32 个通道。

In [ ]:
Pyvium.set_connection_mode(1)  # 对 WE32 切换回标准 EStat

# 选择通道 1 为活跃电极
Pyvium.set_we32_channel(1)
print("WE32 通道 1 已设为活跃")

## B2. 设置偏移

WE32 偏移对各通道施加的电位进行移位（–2 V 到 +2 V）。可让每个电极略有不同的偏置，适用于：
- 补偿阵列各处不同的 OCV
- 运行电位梯度实验

**单通道偏移：**

In [ ]:
# 将通道 3 设置为 +0.1 V 偏移
Pyvium.set_we32_offset(3, 0.1)
print("通道 3 偏移: +0.1 V")

# 使用 channel_index=0 将相同偏移应用到所有通道
Pyvium.set_we32_offset(0, 0.0)  # 将所有通道重置为 0 V
print("所有通道偏移已重置为 0 V")

**批量偏移配置** — 一次为多个通道提供值列表：

In [ ]:
# 使用线性梯度为 8 个通道设置偏移
N_CHANNELS = 8
offsets = [i * 0.05 for i in range(N_CHANNELS)]  # 0 到 0.35 V，步长 50 mV

Pyvium.set_we32_offsets(N_CHANNELS, offsets)
print(f"已应用偏移 (V): {[f'{v:.2f}' for v in offsets]}")

## B3. 读回已配置的偏移

In [ ]:
readback = Pyvium.get_we32_offsets(N_CHANNELS)
print("偏移读回:")
for i, v in enumerate(readback, start=1):
    print(f"  通道 {i:2d}: {v:+.4f} V")

## B4. 32 通道电流同步读出

`read_we32_currents()` 是核心 WE32 测量函数。它返回 32 个电流值的列表，所有通道在同一时刻采样。

> **同步模式限制：** WE32 同步模式下最大采样率为 **10 次/秒**（调用间最小间隔 0.1 s）。每个通道最大承受 ±1 mA。超过这些限制将产生不可靠的读数。进行时间序列采集时，请相应地控制循环节奏（参见 B6）。

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.1)  # 稳定

currents = Pyvium.read_we32_currents()

print(f"WE32 电流（32 通道，单位 A）:")
for i, I in enumerate(currents, start=1):
    print(f"  通道 {i:2d}: {I:.3e} A")

Pyvium.set_cell_off()

## B5. WE32 电流分布可视化

常见用例：将 32 个电流显示为空间分布图（例如基板上的 4×8 电极阵列）。

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.1)

currents = Pyvium.read_we32_currents()
Pyvium.set_cell_off()

currents_uA = np.array(currents) * 1e6  # 转换为 µA

# 重塑为 4 行 × 8 列
grid = currents_uA.reshape(4, 8)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(grid, cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax, label='电流 (µA)')

# 在每个格中标注通道号
for row in range(4):
    for col in range(8):
        ch = row * 8 + col + 1
        ax.text(col, row, f'{ch}\n{grid[row, col]:.1f}',
                ha='center', va='center', fontsize=8)

ax.set_title("WE32 电流分布图（4×8 阵列）")
ax.set_xlabel("列")
ax.set_ylabel("行")
plt.tight_layout()
plt.show()

## B6. 时间分辨 WE32 采集

在循环中调用 `read_we32_currents()` 以构建所有 32 个通道的时间序列。

> 间隔保持 ≥ 0.1 s（同步模式最大 10 次/秒）。循环开销会在 `time.sleep` 之上增加一些时间，因此将 `INTERVAL_S` 设置为略低于目标值以补偿。

In [ ]:
DURATION_S = 5.0
INTERVAL_S = 0.5
CHANNELS_TO_SHOW = [0, 7, 15, 31]  # 从 0 开始的通道索引

Pyvium.set_potential(0.1)
Pyvium.set_cell_on()
time.sleep(0.1)

timeline = []
all_readings = []
t_start = time.time()

while (time.time() - t_start) < DURATION_S:
    t = time.time() - t_start
    snapshot = Pyvium.read_we32_currents()
    timeline.append(t)
    all_readings.append([c * 1e6 for c in snapshot])  # µA
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()

all_readings = np.array(all_readings)  # 形状：(时间点数, 32)

fig, ax = plt.subplots(figsize=(9, 4))
for ch_idx in CHANNELS_TO_SHOW:
    ax.plot(timeline, all_readings[:, ch_idx], label=f'通道 {ch_idx+1}')

ax.set_xlabel("时间 (s)")
ax.set_ylabel("电流 (µA)")
ax.set_title("WE32 — 所选通道随时间变化")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 清理

In [ ]:
Pyvium.set_cell_off()
Pyvium.disconnect_device()
Pyvium.close_driver()
print("完成")

---

## 小结

### BiStat

| 任务 | 方法 |
|------|--------|
| 启用 BiStat | `set_connection_mode(10)` 或 `(11)` |
| 设置 BiStat 模式 / 自动电流量程 | `set_bistat_mode(0 或 1)` |
| 设置 WE2 电位偏移 | `set_we2_potential(V)` |
| 设置 WE2 电流量程 | `set_we2_current_range(n)` |
| 读取 WE2 电流 | `get_we2_current()` → float |

### WE32 *（未经测试 — 无可用硬件）*

| 任务 | 方法 | 状态 |
|------|--------|--------|
| 选择活跃通道 | `set_we32_channel(index)` | 🔬 未测试 |
| 设置单通道偏移 | `set_we32_offset(index, V)` | 🔬 未测试 |
| 设置多通道偏移 | `set_we32_offsets(n, [V, ...])` | 🔬 未测试 |
| 读回已配置偏移 | `get_we32_offsets(n)` → list | 🔬 未测试 |
| 读取所有 32 通道电流 | `read_we32_currents()` → list[32] | 🔬 未测试 |

## 下一步

- **`06_method_mode.ipynb`** — 自动运行 RRDE、EIS 或任何其他方法文件